# Run ChurnLab Straight From GitHub — No Zip Uploads

This notebook replaces the old *"zip `src/`, upload it, unzip it"* workflow.

Instead of uploading the code, it:

1. pulls the latest code from **https://github.com/nikhilwankhedee/churn** (`main` branch),
2. installs any missing dependencies from the repo's `requirements.txt`,
3. adds the cloned repo to `sys.path`, and
4. runs the full pipeline (`run_pipeline`) on your chosen dataset.

**You still attach the *data* datasets as Kaggle inputs** (exactly as before, under `/kaggle/input/datasets`) — only the *code* now comes live from GitHub, so it is always up to date.

> Just hit **Run All**.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/nikhilwankhedee/churn.git"
REPO_BRANCH = "main"
UPDATE_REPO = True            # git pull the latest code on every run

# Run ALL registered datasets at once, or just one?
RUN_ALL = True                # True  -> every registered dataset
                              # False -> only DATASET below
DATASET = "olist"             # used only when RUN_ALL = False
# Available datasets:
#   olist | rees46 | retailrocket | online_retail_ii | instacart | telco | lastfm | credit_card

CHURN_WINDOW_OVERRIDE = None  # None -> dataset default (e.g. 180 days)
USE_SMOTE = False             # balance the training fold with SMOTE
SENSITIVITY = False           # additionally run churn-window sensitivity analysis

# ── Environment / paths ──────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if ON_KAGGLE else ("/content" if os.path.isdir("/content") else os.getcwd())
REPO_DIR = os.path.join(WORK_DIR, "churn")

print(f"Environment  : {'KAGGLE' if ON_KAGGLE else ('COLAB' if '/content' in WORK_DIR else 'LOCAL')}")
print(f"Code will go : {REPO_DIR}")

In [ ]:
# ── Clone / update the code from GitHub ───────────────────────────────────────
import io
import os
import sys
import shutil
import subprocess
import zipfile
import urllib.request


def git(cmd, cwd=None):
    print("$ git " + " ".join(cmd))
    subprocess.check_call(["git"] + cmd, cwd=cwd)


def sync_repo(repo_dir, url, branch, update=True):
    """Clone the repo, or pull the latest code if it already exists.
    Falls back to downloading GitHub's zipball when git is unavailable."""
    repo_dir = os.path.abspath(repo_dir)
    parent = os.path.dirname(repo_dir)

    if os.path.isdir(os.path.join(repo_dir, ".git")):
        if not update:
            print(f"→ Using existing repo at {repo_dir} (UPDATE_REPO=False)")
        else:
            try:
                git(["fetch", "origin"], repo_dir)
                git(["checkout", "--quiet", branch], repo_dir)
                git(["pull", "--rebase", "origin", branch], repo_dir)
                print("→ Repo updated to latest.")
            except Exception as exc:
                # network hiccup — keep the local checkout, it still works offline
                print(f"→ Update failed ({exc}); reusing existing checkout.")
        return repo_dir

    try:
        git(["clone", "--branch", branch, url, repo_dir])
        print(f"→ Cloned {url} → {repo_dir}")
    except Exception:
        # git missing or blocked (e.g. no git binary) — download the zipball instead
        print("git clone failed — downloading GitHub zipball instead…")
        if os.path.isdir(repo_dir):
            shutil.rmtree(repo_dir)  # leftover from a partial clone
        os.makedirs(parent, exist_ok=True)
        zip_url = url.replace(".git", "") + f"/archive/refs/heads/{branch}.zip"
        with urllib.request.urlopen(zip_url, timeout=120) as resp:
            data = resp.read()
        extract_dir = os.path.join(parent, "__churn_extract__")
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            zf.extractall(extract_dir)
        extracted = [os.path.join(extract_dir, d) for d in os.listdir(extract_dir)
                     if os.path.isdir(os.path.join(extract_dir, d))]
        shutil.move(extracted[0], repo_dir)
        shutil.rmtree(extract_dir)
        print(f"→ Downloaded {zip_url} → {repo_dir}")
    return repo_dir


REPO_DIR = sync_repo(REPO_DIR, REPO_URL, REPO_BRANCH, update=UPDATE_REPO)

In [ ]:
# ── Install any missing dependencies (from the repo's requirements.txt) ──────
from importlib.util import find_spec

MODULES = {
    "pandas": "pandas", "numpy": "numpy", "scikit-learn": "sklearn",
    "xgboost": "xgboost", "lightgbm": "lightgbm", "imbalanced-learn": "imblearn",
    "shap": "shap", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "scipy": "scipy", "pingouin": "pingouin", "statsmodels": "statsmodels",
    "joblib": "joblib", "tqdm": "tqdm", "pyyaml": "yaml", "typer": "typer",
    "rich": "rich", "openpyxl": "openpyxl",
}
SKIP = {"jupyter", "ipykernel"}  # already provided by the notebook host

missing = []
with open(os.path.join(REPO_DIR, "requirements.txt")) as fh:
    for line in fh:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name = line.split(">=")[0].split("==")[0].split("[")[0].strip()
        if name.lower() in SKIP:
            continue
        mod = MODULES.get(name.lower(), name.lower().replace("-", "_"))
        if find_spec(mod) is None:
            missing.append(name)

if missing:
    print(f"Installing {len(missing)} missing package(s): {missing}")
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        # PEP 668 'externally managed environment' (e.g. Debian/Ubuntu system
        # python) — retry with the OS-sanctioned override flag
        print("pip refused — retrying with --break-system-packages")
        subprocess.check_call(cmd + ["--break-system-packages"])
else:
    print("All pipeline dependencies are already installed.")

In [ ]:
# ── Add the cloned code to the path and verify the framework imports ─────────
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

from src.config import ON_KAGGLE, PROJECT_ROOT, RESULTS_DIR, FIGURES_DIR, MODELS_DIR, PROCESSED_DIR
from src.datasets import list_datasets

print(f"On Kaggle    : {ON_KAGGLE}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Reg datasets : {', '.join(list_datasets())}")

# version of the code we are running (last commit)
try:
    rev = subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1",
                                   "--format=%h  %cs  %s"], text=True).strip()
    print(f"Code version : {rev}")
except Exception:
    pass

In [ ]:
# ── Run the pipeline ─────────────────────────────────────────────────────────
import pandas as pd
from IPython.display import display
from src.datasets import list_datasets
from src.pipeline import run_pipeline

# "all" -> every dataset registered in the framework
targets = list_datasets() if RUN_ALL else [DATASET]
print(f"Targets : {', '.join(targets)}")
print(f"window  : {CHURN_WINDOW_OVERRIDE or 'default'} | smote={USE_SMOTE} | sensitivity={SENSITIVITY}\n")


def _run_one(name):
    return run_pipeline(
        dataset=name,
        sensitivity=SENSITIVITY,
        churn_window_override=CHURN_WINDOW_OVERRIDE,
        use_smote=USE_SMOTE,
    )


rows, metrics = [], None
for name in targets:
    try:
        result = _run_one(name)
        rows.append({
            "dataset": name,
            "status": "OK",
            "best_model": result.get("best_model"),
            "churn_rate": round(result["churn_rate"], 4) if result.get("churn_rate") is not None else None,
            "imbalance": round(result["imbalance_ratio"], 2) if result.get("imbalance_ratio") is not None else None,
            "seconds": round(result.get("duration_seconds", 0), 1),
        })
        metrics = result.get("metrics")
    except Exception as exc:
        msg = str(exc)
        rows.append({"dataset": name, "status": "FAILED", "best_model": None,
                     "churn_rate": None, "imbalance": None, "seconds": None})
        print(f"[!] {name} FAILED: {msg[:140]}")
        if any(k in msg for k in ("No such file", "FileNotFound", "does not exist", "could not be found")):
            print(f"    Raw data not attached for '{name}' - add its Kaggle input under /kaggle/input/datasets.")

summary = pd.DataFrame(rows)
display(summary)
n_ok = int((summary["status"] == "OK").sum())
print(f"\n{n_ok}/{len(summary)} dataset(s) completed.")

if metrics is not None:
    print("\nPer-model metrics (last completed dataset):")
    display(metrics)

## What just happened

The pipeline ran end-to-end and wrote its artefacts under `PROJECT_ROOT`:

| Folder | Contents |
|---|---|
| `results/` | model metrics, statistical tests, risk scores, data quality & experiments |
| `figures/` | ROC/PR curves, SHAP plots, segmentation, calibration, behavioural insights |
| `models/` | saved fitted models |
| `processed_data/` | cleaned, leakage-free training/validation/test folds |

- On **Kaggle** these live in `/kaggle/working` and persist for the session.
- On **Colab** re-run that cell / notebook to get them after a session restart, or copy them to Drive.
- Changing `DATASET` in the Settings cell and re-running `Run All` runs another dataset. Re-running also **auto-updates the code** (`git pull`).
- With **`RUN_ALL = True`** the run cell executes every registered dataset and prints a summary table (best model, churn rate, runtime per dataset). A dataset whose data isn't attached shows `FAILED` and is skipped without stopping the others.

In [ ]:
# ── Inspect the produced artefacts ────────────────────────────────────────────
print("Output artefacts:")
for d in ("results", "figures", "models", "processed_data"):
    p = os.path.join(PROJECT_ROOT, d)
    if os.path.isdir(p):
        items = sorted(os.listdir(p))
        head = ', '.join(items[:6]) + (" …" if len(items) > 6 else "")
        print(f"  {d}/  ({len(items)} entries)  {head}")
    else:
        print(f"  {d}/  (missing — run the pipeline cell first)")

## Notes

- **CLI equivalent:** the same code ships a `churn` command-line tool. After `!pip install -e {REPO_DIR}` you can run `!churn run all` or `!churn benchmark /kaggle/input` from a shell as an alternative to this notebook.
- **Offline reuse:** the cloned copy at `WORK_DIR/churn` is a normal git repo, so `UPDATE_REPO = False` freezes the code you have and skips the network on re-runs.
- **No more zip uploads:** to update the pipeline you only need to re-run this notebook — git pulls the latest code from https://github.com/nikhilwankhedee/churn automatically.